In [1]:
import contextlib
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
import wandb
from lightning import Fabric
from tqdm.autonotebook import tqdm

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.config.base import BaseConfig
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
)
from src.experiments.multimodal_gaussian.config import MultimodalGaussianTrainingConfig
from src.experiments.multimodal_gaussian.plotting.constraints import (
    plot_latents,
    plot_sample_pairs,
)

torch.set_float32_matmul_precision("medium")

In [2]:
num_modes = 8
mode_std = 0.1
offset = 0.0
latent_dimension = 128
ambient_dimension = 128
experiment_name = f"{datetime.now():%m%d%H%M%S}-{uuid4().hex[:6]}"
wandb_project = "diffusive-latent-generation"

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=mode_std,
    offset=offset,
    ambient_dimension=ambient_dimension,
)

chart_transport_config = get_canonical_chart_transport_configs(
    data_config=data_config,
    latent_dimension=latent_dimension,
)

training_config = MultimodalGaussianTrainingConfig.initialize(
    seed=0,
    train_batch_size=4096,
    eval_batch_size=4096,
    integrated_n_steps=1_000_000,
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / experiment_name
    ),
)

chart_transport_config.visualize()

In [ ]:
class MonitorConfig(BaseConfig):
    constraint_monitor_config: ConstraintMonitorConfig
    critic_monitor_config: CriticMonitorConfig
    integrated_monitor_config: IntegratedMonitorConfig
    log_every_n_steps_constraint_pretrain: int
    log_every_n_steps_critic_pretrain: int
    log_every_n_steps_integrated: int

monitor_config = MonitorConfig(
    constraint_monitor_config=ConstraintMonitorConfig(
        plot_n_sample_pairs=1_000,
        plot_n_data_latents_per_mode=5_000,
    ),
    critic_monitor_config=CriticMonitorConfig(
        sample_t_values=[0.03, 0.2, 0.5],
        num_contour_lines=10,
        plot_n_data_latents_per_mode=5_000,
        plot_n_vectors_per_mode=50,
    ),
    integrated_monitor_config=IntegratedMonitorConfig(
        plot_n_generated_samples=3_000,
        plot_n_data_samples_per_mode=500,
    ),
    log_every_n_steps_constraint_pretrain=1000,
    log_every_n_steps_critic_pretrain=1000,
    log_every_n_steps_integrated=4000,
)
monitor_config.visualize()

In [3]:
fabric = Fabric(
    accelerator="cuda", devices=1,
    precision="bf16-mixed",
)
fabric.launch()
fabric.seed_everything(training_config.seed)

device = fabric.device
device_type = device.type

runtime_data_config = data_config.replace(
    path="isometry",
    replacement=data_config.isometry.to(device=device, dtype=torch.float32),
).replace(
    path="projection",
    replacement=data_config.projection.to(device=device, dtype=torch.float32),
)

architecture_config = chart_transport_config.architecture_config
chart_transport_model = architecture_config.get_model()
optimizer = architecture_config.get_optimizer(
    model=chart_transport_model,
)
chart_transport_model, optimizer = fabric.setup(chart_transport_model, optimizer)

encoder = chart_transport_model.encoder
decoder = chart_transport_model.decoder
critic = chart_transport_model.critic

prior_config = chart_transport_config.prior_config
constraint_config = chart_transport_config.loss_config.constraint_config
constraint_method = constraint_config.constraint_method
pretrain_config = chart_transport_config.loss_config.chart_pretrain_config
transport_config = chart_transport_config.loss_config.transport_config
transport_t_min, transport_t_max = transport_config.t_range

data_dual = torch.zeros((), device=device)
prior_dual = torch.zeros((), device=device)

fabric.print(
    f"device={device}, precision=bf16-mixed, folder={training_config.folder}"
)

wandb_run = wandb.init(
    project=wandb_project,
    name=experiment_name,
    dir=str(training_config.folder),
    tags=["chart-transport", "multimodal-gaussian"],
    config={
        "notebook": "chart-transport.ipynb",
        "seed": training_config.seed,
        "train_batch_size": training_config.train_batch_size,
        "eval_batch_size": training_config.eval_batch_size,
        "integrated_n_steps": training_config.integrated_n_steps,
        "num_modes": num_modes,
        "mode_std": mode_std,
        "offset": offset,
        "latent_dimension": latent_dimension,
        "ambient_dimension": ambient_dimension,
    },
)
for stage_name in ("pretrain", "critic_pretrain", "integrated"):
    wandb.define_metric(f"{stage_name}/local_step")
    wandb.define_metric(f"{stage_name}/*", step_metric=f"{stage_name}/local_step")


Using bfloat16 Automatic Mixed Precision (AMP)


Seed set to 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nlyu/.netrc.
wandb: Currently logged in as: lyuxingjian (lyuxingjian-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


device=cuda:0, precision=bf16-mixed, folder=/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/0329092621-0a1093


## Pretrain

In [7]:
latest_pretrain_metrics = None
encoder.train()
decoder.train()
critic.eval()

pretrain_progress = tqdm(
    range(1, chart_transport_config.scheduling_config.pretrain_chart_n_steps + 1),
    desc="pretrain",
)
for step in pretrain_progress:
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context():
        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        prior_batch = prior_config.sample(
            batch_size=training_config.train_batch_size,
        ).to(device=device, dtype=torch.float32)

        data_latents = encoder(data_batch)
        data_reconstruction = decoder(data_latents)
        prior_reconstruction = decoder(prior_batch)
        prior_latents = encoder(prior_reconstruction)

        data_cycle_loss = F.huber_loss(
            data_reconstruction,
            data_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        prior_cycle_loss = F.huber_loss(
            prior_latents,
            prior_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        constraint_loss = data_cycle_loss + prior_cycle_loss

        zero_mean_loss = F.huber_loss(
            data_latents.mean(),
            torch.zeros((), device=device, dtype=data_latents.dtype),
            delta=1.0,
            reduction="mean",
        )
        latent_norms = data_latents.norm(dim=-1)
        latent_norm_loss = F.huber_loss(
            latent_norms,
            torch.zeros_like(latent_norms),
            delta=pretrain_config.latent_norm_delta,
            reduction="mean",
        )
        chart_loss = constraint_loss
        chart_loss = chart_loss + pretrain_config.zero_mean_weight * zero_mean_loss
        chart_loss = chart_loss + pretrain_config.latent_norm_weight * latent_norm_loss

    fabric.backward(chart_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
        error_if_nonfinite=False
    )
    optimizer.step()

    metrics = {
        "stage": "pretrain",
        "step": step,
        "chart_loss": chart_loss.detach().item(),
        "data_cycle_loss": data_cycle_loss.detach().item(),
        "prior_cycle_loss": prior_cycle_loss.detach().item(),
        "zero_mean_loss": zero_mean_loss.detach().item(),
        "latent_norm_loss": latent_norm_loss.detach().item(),
    }
    latest_pretrain_metrics = metrics
    log_wandb_scalars(stage="pretrain", step=step, metrics=metrics)
    pretrain_progress.set_postfix(
        chart_loss=f"{metrics['chart_loss']:.4f}",
        data_cycle=f"{metrics['data_cycle_loss']:.4f}",
        prior_cycle=f"{metrics['prior_cycle_loss']:.4f}",
    )

    if should_log_monitor(
        step=step,
        total_steps=chart_transport_config.scheduling_config.pretrain_chart_n_steps,
        every_n_steps=monitor_config.log_every_n_steps_constraint_pretrain,
    ):
        fabric.print(
            f"[pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_chart_n_steps}: {format_metrics_summary(metrics)}"
        )

        encoder_was_training = encoder.training
        decoder_was_training = decoder.training
        encoder.eval()
        decoder.eval()

        with torch.no_grad():
            with runtime_precision_context():
                pair_samples, pair_labels = sample_monitor_pairs_batch(
                    total_batch_size=monitor_config.constraint_monitor_config.plot_n_sample_pairs,
                )
                pair_latents = encoder(pair_samples)
                pair_reconstructions = decoder(pair_latents)

                latent_samples, latent_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.constraint_monitor_config.plot_n_data_latents_per_mode,
                )
                latent_values = encoder(latent_samples)

            pair_samples_cpu = pair_samples.detach().cpu().float()
            pair_reconstructions_cpu = pair_reconstructions.detach().cpu().float()
            pair_labels_cpu = pair_labels.detach().cpu().long()
            pair_projected_samples, _, _ = data_config.decompose_projection(
                pair_samples_cpu,
            )
            pair_projected_reconstructions, _, _ = data_config.decompose_projection(
                pair_reconstructions_cpu,
            )
            pair_manifold_deviation = (
                (pair_reconstructions - pair_samples)
                .reshape(pair_samples.shape[0], -1)
                .norm(dim=-1)
                .detach()
                .cpu()
                .float()
            )

            latent_samples_cpu = latent_samples.detach().cpu().float()
            latent_values_cpu = latent_values.detach().cpu().float()
            latent_labels_cpu = latent_labels.detach().cpu().long()
            _, _, latent_off_plane = data_config.decompose_projection(
                latent_samples_cpu,
            )
            latent_off_manifold_norm = latent_off_plane.norm(dim=-1).float()

        if encoder_was_training:
            encoder.train()
        if decoder_was_training:
            decoder.train()

        write_constraint_monitor_artifacts(
            step=step,
            stage="pretrain",
            plot_prefix="",
            pair_projected_samples=pair_projected_samples,
            pair_projected_reconstructions=pair_projected_reconstructions,
            pair_manifold_deviation=pair_manifold_deviation,
            pair_labels=pair_labels_cpu,
            latent_values=latent_values_cpu,
            latent_off_manifold_norm=latent_off_manifold_norm,
            latent_labels=latent_labels_cpu,
        )

latest_pretrain_metrics


pretrain:   0%|          | 0/2000 [00:00<?, ?it/s]

[pretrain] step 1/2000: chart_loss=0.5822, data_cycle_loss=0.0601, prior_cycle_loss=0.5215, zero_mean_loss=0.0000, latent_norm_loss=6.7365
[pretrain] step 1000/2000: chart_loss=0.0012, data_cycle_loss=0.0002, prior_cycle_loss=0.0009, zero_mean_loss=0.0000, latent_norm_loss=0.8899
[pretrain] step 2000/2000: chart_loss=0.0028, data_cycle_loss=0.0005, prior_cycle_loss=0.0021, zero_mean_loss=0.0000, latent_norm_loss=2.7123


{'stage': 'pretrain',
 'step': 2000,
 'chart_loss': 0.0027891991194337606,
 'data_cycle_loss': 0.00045837811194360256,
 'prior_cycle_loss': 0.0020595923997461796,
 'zero_mean_loss': 4.610163159668446e-06,
 'latent_norm_loss': 2.7122859954833984}

## Train noise critic

In [8]:
latest_critic_pretrain_metrics = None
encoder.eval()
decoder.eval()
critic.train()

critic_pretrain_progress = tqdm(
    range(1, chart_transport_config.scheduling_config.pretrain_critic_n_steps + 1),
    desc="critic_pretrain",
)
for step in critic_pretrain_progress:
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context():
        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            data_latents = encoder(data_batch)

        t = sample_transport_times(
            batch_shape=(data_latents.shape[0],),
        )
        eps = torch.randn_like(data_latents)
        noised_latents = (
            (1.0 - t).unsqueeze(-1) * data_latents + t.unsqueeze(-1) * eps
        )
        predicted_noise = critic(noised_latents, t)
        critic_loss = F.mse_loss(predicted_noise, eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "critic_pretrain",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
    }
    latest_critic_pretrain_metrics = metrics
    log_wandb_scalars(stage="critic_pretrain", step=step, metrics=metrics)
    critic_pretrain_progress.set_postfix(
        critic_loss=f"{metrics['critic_loss']:.4f}",
    )

    if should_log_monitor(
        step=step,
        total_steps=chart_transport_config.scheduling_config.pretrain_critic_n_steps,
        every_n_steps=monitor_config.log_every_n_steps_critic_pretrain,
    ):
        fabric.print(
            f"[critic_pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_critic_n_steps}: {format_metrics_summary(metrics)}"
        )

        encoder_was_training = encoder.training
        critic_was_training = critic.training
        encoder.eval()
        critic.eval()

        with torch.no_grad():
            with runtime_precision_context():
                dense_samples, dense_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_data_latents_per_mode,
                )
                vector_samples, vector_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_vectors_per_mode,
                )
                dense_clean_latents = encoder(dense_samples).float()
                vector_clean_latents = encoder(vector_samples).float()

                score_snapshots = []
                for t_value in monitor_config.critic_monitor_config.sample_t_values:
                    cloud_latents, _ = sample_critic_snapshot(
                        clean_latents=dense_clean_latents,
                        t_value=t_value,
                    )
                    arrow_latents, data_score = sample_critic_snapshot(
                        clean_latents=vector_clean_latents,
                        t_value=t_value,
                    )
                    score_snapshots.append(
                        (
                            t_value,
                            cloud_latents.detach().cpu().float(),
                            arrow_latents.detach().cpu().float(),
                            data_score.detach().cpu().float(),
                        )
                    )

                spectrum_t_values = critic_spectrum_t_values()
                transport_field = estimate_clean_transport_field(
                    clean_latents=vector_clean_latents,
                    t_values=spectrum_t_values,
                )
                transport_grid_points, transport_grid_xs, transport_grid_ys = build_latent_grid(
                    points=dense_clean_latents,
                    resolution=31,
                )
                transport_grid_field = estimate_clean_transport_field(
                    clean_latents=transport_grid_points,
                    t_values=spectrum_t_values,
                )
                transport_grid_projection = project_latent_vectors_to_pca_plane(
                    reference_points=dense_clean_latents,
                    vectors=transport_grid_field,
                ).norm(dim=-1).reshape(
                    transport_grid_ys.shape[0],
                    transport_grid_xs.shape[0],
                )

            dense_clean_latents_cpu = dense_clean_latents.detach().cpu().float()
            dense_labels_cpu = dense_labels.detach().cpu().long()
            vector_clean_latents_cpu = vector_clean_latents.detach().cpu().float()
            vector_labels_cpu = vector_labels.detach().cpu().long()
            transport_field_cpu = transport_field.detach().cpu().float()
            transport_grid_xs_cpu = transport_grid_xs.detach().cpu().float()
            transport_grid_ys_cpu = transport_grid_ys.detach().cpu().float()
            transport_grid_projection_cpu = (
                transport_grid_projection.detach().cpu().float()
            )

        if encoder_was_training:
            encoder.train()
        if critic_was_training:
            critic.train()

        write_critic_monitor_artifacts(
            step=step,
            stage="critic_pretrain",
            plot_prefix="",
            dense_clean_latents=dense_clean_latents_cpu,
            dense_labels=dense_labels_cpu,
            vector_clean_latents=vector_clean_latents_cpu,
            vector_labels=vector_labels_cpu,
            score_snapshots=score_snapshots,
            transport_field=transport_field_cpu,
            transport_grid_xs=transport_grid_xs_cpu,
            transport_grid_ys=transport_grid_ys_cpu,
            transport_grid_projection=transport_grid_projection_cpu,
            num_contour_lines=monitor_config.critic_monitor_config.num_contour_lines,
        )

latest_critic_pretrain_metrics


critic_pretrain:   0%|          | 0/2000 [00:00<?, ?it/s]

[critic_pretrain] step 1/2000: critic_loss=1.1226
[critic_pretrain] step 1000/2000: critic_loss=0.0095
[critic_pretrain] step 2000/2000: critic_loss=0.0077


{'stage': 'critic_pretrain', 'step': 2000, 'critic_loss': 0.007691949140280485}

## Integrated training

In [ ]:
latest_integrated_metrics = None
critic_steps_per_chart_step = max(
    1,
    chart_transport_config.scheduling_config.update_chart_every_n_critic_steps,
)
latest_chart_metrics = {
    "chart_loss": float("nan"),
    "encoder_transport_loss": float("nan"),
    "decoder_transport_loss": float("nan"),
    "transport_field_norm": float("nan"),
    "avg_generated_log_likelihood": float("nan"),
    "data_cycle_loss": float("nan"),
    "prior_cycle_loss": float("nan"),
}

integrated_progress = tqdm(
    range(1, training_config.integrated_n_steps + 1),
    desc="integrated",
)
for step in integrated_progress:
    encoder.eval()
    decoder.eval()
    critic.train()
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context():
        critic_data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            critic_data_latents = encoder(critic_data_batch)

        critic_t = sample_transport_times(
            batch_shape=(critic_data_latents.shape[0],),
        )
        critic_eps = torch.randn_like(critic_data_latents)
        critic_noised_latents = (
            (1.0 - critic_t).unsqueeze(-1) * critic_data_latents
            + critic_t.unsqueeze(-1) * critic_eps
        )
        critic_predicted_noise = critic(critic_noised_latents, critic_t)
        critic_loss = F.mse_loss(critic_predicted_noise, critic_eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "integrated",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
        **latest_chart_metrics,
        "data_dual": data_dual.detach().item(),
        "prior_dual": prior_dual.detach().item(),
    }
    wandb_metrics = {
        "critic_loss": critic_loss.detach().item(),
        "data_dual": data_dual.detach().item(),
        "prior_dual": prior_dual.detach().item(),
    }

    if step == 1 or step % critic_steps_per_chart_step == 0:
        encoder.train()
        decoder.train()
        optimizer.zero_grad(set_to_none=True)

        with runtime_precision_context():
            chart_data_batch = runtime_data_config.sample_unconditional(
                batch_size=training_config.train_batch_size,
            )
            chart_prior_batch = prior_config.sample(
                batch_size=training_config.train_batch_size,
            ).to(device=device, dtype=torch.float32)

            chart_data_latents = encoder(chart_data_batch)
            chart_data_reconstruction = decoder(chart_data_latents)
            chart_prior_reconstruction = decoder(chart_prior_batch)
            chart_prior_latents = encoder(chart_prior_reconstruction)

            data_cycle_loss = F.huber_loss(
                chart_data_reconstruction,
                chart_data_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )
            prior_cycle_loss = F.huber_loss(
                chart_prior_latents,
                chart_prior_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )

            if isinstance(constraint_method, LossConstraintConfig):
                constraint_loss = (
                    constraint_method.data_loss_weight * data_cycle_loss
                    + constraint_method.prior_loss_weight * prior_cycle_loss
                )
            else:
                constraint_loss = data_cycle_loss + prior_cycle_loss
                constraint_loss = constraint_loss + data_dual * (
                    data_cycle_loss - constraint_method.data_constraint_budget
                )
                constraint_loss = constraint_loss + prior_dual * (
                    prior_cycle_loss - constraint_method.prior_constraint_budget
                )

            with torch.no_grad():
                transport_source_latents = chart_data_latents.detach()
                transport_t = sample_stratified_transport_times(
                    batch_size=transport_source_latents.shape[0],
                    num_time_samples=transport_config.num_time_samples,
                )

                transport_source_latents = transport_source_latents.unsqueeze(1).expand(
                    -1,
                    transport_config.num_time_samples,
                    -1,
                )
                transport_eps = torch.randn(
                    transport_source_latents.shape[0],
                    transport_config.num_time_samples,
                    transport_source_latents.shape[-1],
                    device=device,
                    dtype=transport_source_latents.dtype,
                )

                if transport_config.antipodal_estimate:
                    transport_t = torch.cat([transport_t, transport_t], dim=1)
                    transport_source_latents = transport_source_latents.repeat(1, 2, 1)
                    transport_eps = torch.cat([transport_eps, -transport_eps], dim=1)

                transport_noised_latents = (
                    (1.0 - transport_t).unsqueeze(-1) * transport_source_latents
                    + transport_t.unsqueeze(-1) * transport_eps
                )
                flat_transport_noised_latents = transport_noised_latents.reshape(
                    -1,
                    transport_noised_latents.shape[-1],
                )
                flat_transport_t = transport_t.reshape(-1)

                transport_predicted_noise = critic(
                    flat_transport_noised_latents,
                    flat_transport_t,
                ).reshape_as(transport_noised_latents)
                transport_prior_score = prior_config.analytic_score(
                    flat_transport_noised_latents.float(),
                    flat_transport_t.float(),
                ).to(dtype=flat_transport_noised_latents.dtype).reshape_as(
                    transport_noised_latents
                )
                transport_pullback_weight = (
                    transport_config.kl_weight_schedule.pullback_weight(
                        flat_transport_t.float(),
                    )
                    .to(dtype=flat_transport_noised_latents.dtype)
                    .reshape_as(transport_t)
                )
                transport_field_terms = transport_pullback_weight.unsqueeze(-1) * (
                    transport_prior_score
                    + transport_predicted_noise / transport_t.unsqueeze(-1)
                )

                if transport_config.antipodal_estimate:
                    midpoint = transport_config.num_time_samples
                    transport_field_terms = 0.5 * (
                        transport_field_terms[:, :midpoint]
                        + transport_field_terms[:, midpoint:]
                    )

                transport_field = transport_field_terms.mean(dim=1)
                transport_field_norm = transport_field.norm(dim=-1, keepdim=True)
                transport_step_size = torch.minimum(
                    torch.full_like(
                        transport_field_norm,
                        transport_config.transport_step_size,
                    ),
                    torch.full_like(
                        transport_field_norm,
                        transport_config.transport_step_cap,
                    )
                    / transport_field_norm.clamp_min(1e-6),
                )
                transport_step = transport_step_size * transport_field
                transported_latents = chart_data_latents.detach() + transport_step

            encoder_transport_loss = F.mse_loss(
                chart_data_latents,
                transported_latents.detach(),
            )
            decoder_transport_loss = F.mse_loss(
                decoder(transported_latents),
                chart_data_batch.detach(),
            )
            generated_log_likelihood = runtime_data_config.log_likelihood(
                chart_prior_reconstruction.float(),
            ).mean()
            chart_loss = constraint_loss
            chart_loss = chart_loss + (
                transport_config.encoder_transport_weight * encoder_transport_loss
            )
            chart_loss = chart_loss + (
                transport_config.decoder_transport_weight * decoder_transport_loss
            )

        fabric.backward(chart_loss)
        fabric.clip_gradients(
            chart_transport_model,
            optimizer,
            max_norm=architecture_config.grad_clip_norm,
        )
        optimizer.step()

        if isinstance(constraint_method, LagrangianConstraintConfig):
            data_dual = (
                data_dual
                + constraint_method.dual_variable_lr
                * (data_cycle_loss.detach() - constraint_method.data_constraint_budget)
            ).clamp_min(0.0)
            prior_dual = (
                prior_dual
                + constraint_method.dual_variable_lr
                * (prior_cycle_loss.detach() - constraint_method.prior_constraint_budget)
            ).clamp_min(0.0)

        latest_chart_metrics = {
            "chart_loss": chart_loss.detach().item(),
            "encoder_transport_loss": encoder_transport_loss.detach().item(),
            "decoder_transport_loss": decoder_transport_loss.detach().item(),
            "transport_field_norm": transport_field_norm.mean().detach().item(),
            "avg_generated_log_likelihood": generated_log_likelihood.detach().item(),
            "data_cycle_loss": data_cycle_loss.detach().item(),
            "prior_cycle_loss": prior_cycle_loss.detach().item(),
        }
        metrics.update(latest_chart_metrics)
        metrics["data_dual"] = data_dual.detach().item()
        metrics["prior_dual"] = prior_dual.detach().item()
        wandb_metrics.update(latest_chart_metrics)
        wandb_metrics["data_dual"] = data_dual.detach().item()
        wandb_metrics["prior_dual"] = prior_dual.detach().item()

    latest_integrated_metrics = metrics
    log_wandb_scalars(stage="integrated", step=step, metrics=wandb_metrics)
    integrated_progress.set_postfix(
        critic_loss=f"{metrics['critic_loss']:.4f}",
        chart_loss=f"{metrics['chart_loss']:.4f}",
    )

    if should_log_monitor(
        step=step,
        total_steps=training_config.integrated_n_steps,
        every_n_steps=monitor_config.log_every_n_steps_integrated,
    ):
        fabric.print(
            f"[integrated] step {step}/{training_config.integrated_n_steps}: {format_metrics_summary(metrics)}"
        )

        encoder_was_training = encoder.training
        decoder_was_training = decoder.training
        critic_was_training = critic.training
        encoder.eval()
        decoder.eval()
        critic.eval()

        with torch.no_grad():
            with runtime_precision_context():
                pair_samples, pair_labels = sample_monitor_pairs_batch(
                    total_batch_size=monitor_config.constraint_monitor_config.plot_n_sample_pairs,
                )
                pair_latents = encoder(pair_samples)
                pair_reconstructions = decoder(pair_latents)

                latent_samples, latent_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.constraint_monitor_config.plot_n_data_latents_per_mode,
                )
                latent_values = encoder(latent_samples)

                dense_samples, dense_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_data_latents_per_mode,
                )
                vector_samples, vector_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.critic_monitor_config.plot_n_vectors_per_mode,
                )
                dense_clean_latents = encoder(dense_samples).float()
                vector_clean_latents = encoder(vector_samples).float()

                score_snapshots = []
                for t_value in monitor_config.critic_monitor_config.sample_t_values:
                    cloud_latents, _ = sample_critic_snapshot(
                        clean_latents=dense_clean_latents,
                        t_value=t_value,
                    )
                    arrow_latents, data_score = sample_critic_snapshot(
                        clean_latents=vector_clean_latents,
                        t_value=t_value,
                    )
                    score_snapshots.append(
                        (
                            t_value,
                            cloud_latents.detach().cpu().float(),
                            arrow_latents.detach().cpu().float(),
                            data_score.detach().cpu().float(),
                        )
                    )

                spectrum_t_values = critic_spectrum_t_values()
                transport_field = estimate_clean_transport_field(
                    clean_latents=vector_clean_latents,
                    t_values=spectrum_t_values,
                )
                transport_grid_points, transport_grid_xs, transport_grid_ys = build_latent_grid(
                    points=dense_clean_latents,
                    resolution=31,
                )
                transport_grid_field = estimate_clean_transport_field(
                    clean_latents=transport_grid_points,
                    t_values=spectrum_t_values,
                )
                transport_grid_projection = project_latent_vectors_to_pca_plane(
                    reference_points=dense_clean_latents,
                    vectors=transport_grid_field,
                ).norm(dim=-1).reshape(
                    transport_grid_ys.shape[0],
                    transport_grid_xs.shape[0],
                )

                generated_prior_samples = prior_config.sample(
                    batch_size=monitor_config.integrated_monitor_config.plot_n_generated_samples,
                ).to(device=device, dtype=torch.float32)
                generated_samples = decoder(generated_prior_samples)
                generated_data_samples, generated_data_labels = sample_monitor_batch(
                    batch_size_per_mode=monitor_config.integrated_monitor_config.plot_n_data_samples_per_mode,
                )

            pair_samples_cpu = pair_samples.detach().cpu().float()
            pair_reconstructions_cpu = pair_reconstructions.detach().cpu().float()
            pair_labels_cpu = pair_labels.detach().cpu().long()
            pair_projected_samples, _, _ = data_config.decompose_projection(
                pair_samples_cpu,
            )
            pair_projected_reconstructions, _, _ = data_config.decompose_projection(
                pair_reconstructions_cpu,
            )
            pair_manifold_deviation = (
                (pair_reconstructions - pair_samples)
                .reshape(pair_samples.shape[0], -1)
                .norm(dim=-1)
                .detach()
                .cpu()
                .float()
            )

            latent_samples_cpu = latent_samples.detach().cpu().float()
            latent_values_cpu = latent_values.detach().cpu().float()
            latent_labels_cpu = latent_labels.detach().cpu().long()
            _, _, latent_off_plane = data_config.decompose_projection(
                latent_samples_cpu,
            )
            latent_off_manifold_norm = latent_off_plane.norm(dim=-1).float()

            dense_clean_latents_cpu = dense_clean_latents.detach().cpu().float()
            dense_labels_cpu = dense_labels.detach().cpu().long()
            vector_clean_latents_cpu = vector_clean_latents.detach().cpu().float()
            vector_labels_cpu = vector_labels.detach().cpu().long()
            transport_field_cpu = transport_field.detach().cpu().float()
            transport_grid_xs_cpu = transport_grid_xs.detach().cpu().float()
            transport_grid_ys_cpu = transport_grid_ys.detach().cpu().float()
            transport_grid_projection_cpu = (
                transport_grid_projection.detach().cpu().float()
            )
            generated_samples_projected, _, _ = data_config.decompose_projection(
                generated_samples.detach().cpu().float(),
            )
            generated_data_samples_projected, _, _ = data_config.decompose_projection(
                generated_data_samples.detach().cpu().float(),
            )
            generated_data_labels_cpu = generated_data_labels.detach().cpu().long()

        if encoder_was_training:
            encoder.train()
        if decoder_was_training:
            decoder.train()
        if critic_was_training:
            critic.train()

        write_constraint_monitor_artifacts(
            step=step,
            stage="integrated",
            plot_prefix="constraint_",
            pair_projected_samples=pair_projected_samples,
            pair_projected_reconstructions=pair_projected_reconstructions,
            pair_manifold_deviation=pair_manifold_deviation,
            pair_labels=pair_labels_cpu,
            latent_values=latent_values_cpu,
            latent_off_manifold_norm=latent_off_manifold_norm,
            latent_labels=latent_labels_cpu,
        )
        write_critic_monitor_artifacts(
            step=step,
            stage="integrated",
            plot_prefix="critic_",
            dense_clean_latents=dense_clean_latents_cpu,
            dense_labels=dense_labels_cpu,
            vector_clean_latents=vector_clean_latents_cpu,
            vector_labels=vector_labels_cpu,
            score_snapshots=score_snapshots,
            transport_field=transport_field_cpu,
            transport_grid_xs=transport_grid_xs_cpu,
            transport_grid_ys=transport_grid_ys_cpu,
            transport_grid_projection=transport_grid_projection_cpu,
            num_contour_lines=monitor_config.critic_monitor_config.num_contour_lines,
            score_snapshot_subdir="score_snapshots",
        )
        write_integrated_monitor_artifacts(
            step=step,
            stage="integrated",
            generated_samples=generated_samples_projected,
            generated_data_samples=generated_data_samples_projected,
            generated_data_labels=generated_data_labels_cpu,
        )

wandb.finish()
latest_integrated_metrics


integrated:   0%|          | 0/1000000 [00:00<?, ?it/s]

[integrated] step 1/1000000: critic_loss=0.0074, chart_loss=0.0034, encoder_transport_loss=0.0000, decoder_transport_loss=0.0011, transport_field_norm=5.7629, avg_generated_log_likelihood=-249.8454, data_cycle_loss=0.0005, prior_cycle_loss=0.0018, data_dual=0.0000, prior_dual=0.0000
[integrated] step 4000/1000000: critic_loss=0.0232, chart_loss=0.0013, encoder_transport_loss=0.0000, decoder_transport_loss=0.0005, transport_field_norm=4.6382, avg_generated_log_likelihood=-160.9734, data_cycle_loss=0.0002, prior_cycle_loss=0.0006, data_dual=0.0000, prior_dual=0.0000
[integrated] step 8000/1000000: critic_loss=0.0152, chart_loss=0.0006, encoder_transport_loss=0.0000, decoder_transport_loss=0.0001, transport_field_norm=3.5950, avg_generated_log_likelihood=-28.5944, data_cycle_loss=0.0000, prior_cycle_loss=0.0004, data_dual=0.0000, prior_dual=0.0000
[integrated] step 12000/1000000: critic_loss=0.0193, chart_loss=0.0005, encoder_transport_loss=0.0000, decoder_transport_loss=0.0001, transport